# LoRA Training Template

This notebook helps you get started with **LoRA (Low-Rank Adaptation)** fine-tuning using Training Hub.
LoRA is parameter-efficient: it trains small adapter matrices instead of the full model,
requiring significantly less VRAM.

## What you need
1. A HuggingFace model path (or local model)
2. A training dataset in JSONL chat format
3. At least one GPU

## What this notebook does
- Detects available GPUs and VRAM
- Calculates safe LoRA rank and whether to use QLoRA (4-bit quantization)
- Provides sensible defaults for learning rate, batch size, and epochs
- Launches training with a single call to 

## Setup

In [ ]:
# Install training-hub with LoRA support
# Skip if already installed
# %pip install training-hub[lora]

## 1. User Inputs

Configure these values. Everything else is calculated automatically.

In [ ]:
# === REQUIRED: Set these ===
MODEL_PATH = "ibm-granite/granite-3.3-8b-instruct"  # HuggingFace model or local path
DATA_PATH = "./train_data.jsonl"                     # Path to your JSONL training data
CKPT_OUTPUT_DIR = "./lora_checkpoints"               # Where to save checkpoints

# === OPTIONAL: Hardware overrides ===
NUM_GPUS = None       # Auto-detected if None (LoRA typically uses 1 GPU)
NUM_NODES = 1         # Number of nodes (default: 1)
VRAM_PER_GPU = None   # Auto-detected if None (in GB, e.g. 80)

## 2. Hardware Detection

In [ ]:
import torch
from transformers import AutoConfig

# Detect GPUs
if NUM_GPUS is None:
    NUM_GPUS = torch.cuda.device_count()
    assert NUM_GPUS > 0, "No GPUs detected. Set NUM_GPUS manually."

# Detect VRAM
if VRAM_PER_GPU is None:
    props = torch.cuda.get_device_properties(0)
    VRAM_PER_GPU = props.total_memory / (1024**3)

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"GPUs: {NUM_GPUS}")
print(f"VRAM per GPU: {VRAM_PER_GPU:.1f} GB")

## 3. Calculate Safe Hyperparameters

LoRA memory is dominated by the base model weights (loaded in FP16 or 4-bit)
plus the small adapter matrices. Key decisions:
- **Model fits in FP16?** -> standard LoRA
- **Model too large for FP16?** -> QLoRA (4-bit quantization)
- **LoRA rank**: higher = more capacity but more memory. Start at 16.

In [ ]:
# Load model config
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
num_params = getattr(config, "num_parameters", None)

if num_params is None:
    hidden = config.hidden_size
    vocab = config.vocab_size
    layers = config.num_hidden_layers
    intermediate = getattr(config, "intermediate_size", hidden * 4)
    n_heads = config.num_attention_heads
    n_kv = getattr(config, "num_key_value_heads", n_heads)
    head_dim = hidden // n_heads
    # Attention (GQA-aware) + MLP (gate/up/down) per layer + embeddings
    attn = hidden * (n_heads + 2 * n_kv) * head_dim + hidden * hidden
    mlp = 3 * hidden * intermediate
    num_params = vocab * hidden + layers * (attn + mlp)
# LoRA loads model in FP16 (2 bytes/param) or 4-bit (0.5 bytes/param)
fp16_model_gb = (num_params * 2) / (1024**3)
int4_model_gb = (num_params * 0.5) / (1024**3)

# Decide: use QLoRA if FP16 model takes more than 70% of single GPU VRAM
USE_QLORA = fp16_model_gb > (VRAM_PER_GPU * 0.7)
model_memory_gb = int4_model_gb if USE_QLORA else fp16_model_gb

# Remaining VRAM for adapters + activations
remaining_gb = VRAM_PER_GPU * 0.85 - model_memory_gb

# LoRA rank: start at 16, scale up if we have headroom
if remaining_gb > 20:
    LORA_R = 64
elif remaining_gb > 10:
    LORA_R = 32
else:
    LORA_R = 16

LORA_ALPHA = LORA_R * 2  # Standard: alpha = 2x rank
LORA_DROPOUT = 0.0       # Optimized for Unsloth

# Sequence length
SAFE_MAX_SEQ_LEN = 2048 if remaining_gb < 10 else 4096
MICRO_BATCH_SIZE = 2

strategy = "QLoRA (4-bit)" if USE_QLORA else "LoRA (FP16)"
print(f"
Model: {MODEL_PATH}")
print(f"Estimated parameters: {num_params / 1e9:.2f}B")
print(f"FP16 model size: {fp16_model_gb:.1f} GB")
print(f"INT4 model size: {int4_model_gb:.1f} GB")
print(f"
Strategy: {strategy}")
print(f"Model memory: {model_memory_gb:.1f} GB")
print(f"Remaining for adapters + activations: {remaining_gb:.1f} GB")
print(f"
Calculated safe values:")
print(f"  lora_r = {LORA_R}")
print(f"  lora_alpha = {LORA_ALPHA}")
print(f"  load_in_4bit = {USE_QLORA}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  micro_batch_size = {MICRO_BATCH_SIZE}")

## 4. Training Configuration

In [ ]:
# Training hyperparameters
LEARNING_RATE = 1e-5          # LoRA typically uses higher LR than SFT; up to 1e-4
NUM_EPOCHS = 3

print("Training config:")
print(f"  learning_rate = {LEARNING_RATE}")
print(f"  num_epochs = {NUM_EPOCHS}")
print(f"  lora_r = {LORA_R}")
print(f"  lora_alpha = {LORA_ALPHA}")
print(f"  load_in_4bit = {USE_QLORA}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  micro_batch_size = {MICRO_BATCH_SIZE}")

## 5. Launch Training

In [ ]:
from training_hub import lora_sft

result = lora_sft(
    model_path=MODEL_PATH,
    data_path=DATA_PATH,
    ckpt_output_dir=CKPT_OUTPUT_DIR,
    # LoRA configuration
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    # QLoRA
    load_in_4bit=USE_QLORA,
    # Training hyperparameters
    learning_rate=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    max_seq_len=SAFE_MAX_SEQ_LEN,
    micro_batch_size=MICRO_BATCH_SIZE,
    # Multi-GPU (optional)
    nproc_per_node=NUM_GPUS if NUM_GPUS > 1 else None,
)

print("Training complete!")

## 6. Test the Model

In [ ]:
from unsloth import FastLanguageModel

model = result["model"]
tokenizer = result["tokenizer"]
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Hello! What can you help me with?"}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
outputs = model.generate(inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Tips

- **OOM?** Enable QLoRA (), reduce , or reduce 
- **Underfitting?** Increase  (32, 64) for more capacity
- **Large model on single GPU?** Use  instead of multi-GPU DDP
- **Multiple GPUs?** Set  -- Unsloth uses data-parallel (DDP) by default
- **Better quality?** Try  or increase 